In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [ ]:
def load_yaml(path):
    with open(path, "r") as file:
        return yaml.safe_load(file)


def check_file_exists(path, label=None):
    path = Path(path)

    if not path.exists():
        message = f"Missing path: {path}"
        if label is not None:
            message = f"Missing path for {label}: {path}"
        raise FileNotFoundError(message)

    return path

In [ ]:

def find_h5_file(data_root, sample_id, h5_pattern):
    """Find one filtered h5 file for a sample inside a common directory."""
    data_root = Path(data_root)
    pattern = h5_pattern.format(sample_id=sample_id)

    matches = sorted(data_root.glob(pattern))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No h5 file found for sample '{sample_id}' using pattern: {data_root / pattern}"
        )

    if len(matches) > 1:
        raise ValueError(
            f"Multiple h5 files found for sample '{sample_id}':\n"
            + "\n".join(str(path) for path in matches)
        )

    return matches[0]


In [ ]:


def make_dir(path):
    """Create a directory if it does not exist."""
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

config = load_yaml(PROJECT_DIR / "configs" / "config.yaml")
samples_config = load_yaml(PROJECT_DIR / "configs" / "samples.yaml")

DATA_ROOT = Path(config["paths"]["data_root"])
H5_PATTERN = config["input"]["h5_pattern"]

RESULTS_DIR = make_dir(PROJECT_DIR / config["paths"]["results_dir"])
FIGURES_DIR = make_dir(PROJECT_DIR / config["paths"]["figures_dir"])
ANNDATA_DIR = make_dir(PROJECT_DIR / config["paths"]["anndata_dir"])
TABLES_DIR = make_dir(PROJECT_DIR / config["paths"]["tables_dir"])

sc.settings.figdir = str(FIGURES_DIR)

sample_ids = samples_config["samples"]

metadata = pd.read_csv(PROJECT_DIR / "configs" / "metadata.csv")
metadata = metadata.set_index("sample_id")

metadata

In [ ]:
missing_metadata = set(sample_ids) - set(metadata.index)

if missing_metadata:
    raise ValueError(f"Missing metadata for samples: {missing_metadata}")

metadata = metadata.loc[sample_ids]

samples = metadata.copy()

samples["h5_path"] = [
    find_h5_file(DATA_ROOT, sample_id, H5_PATTERN)
    for sample_id in samples.index
]

samples



In [ ]:


adatas = []

for sample_id, row in samples.iterrows():
    print(f"Reading: {sample_id}")
    print(f"Path: {row['h5_path']}")

    adata_sample = sc.read_10x_h5(
        row["h5_path"],
        genome=None,
        gex_only=True
    )

    adata_sample.var_names_make_unique()

    adata_sample.obs["sample_id"] = sample_id
    adata_sample.obs["bio_sample"] = row["bio_sample"]
    adata_sample.obs["cf_status"] = row["cf_status"]
    adata_sample.obs["condition"] = row["condition"]
    adata_sample.obs["treatment"] = row["treatment"]
    adata_sample.obs["sample_ref"] = row["sample_ref"]

    adata_sample.obs_names = [
        f"{sample_id}_{barcode}" for barcode in adata_sample.obs_names
    ]

    adatas.append(adata_sample)

In [ ]:
adatas

In [ ]:
adata = ad.concat(
    adatas,
    join="outer",
    index_unique=None
)

adata.obs_names_make_unique()

for col in ["sample_id", "bio_sample", "cf_status", "condition", "treatment", "sample_ref"]:
    adata.obs[col] = adata.obs[col].astype("category")

adata

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

adata.var[["mt", "ribo"]].sum()

In [ ]:
def rotate_x_labels(rotation=45):
    """Rotate x-axis labels in the current matplotlib figure."""
    fig = plt.gcf()

    for ax in fig.axes:
        ax.tick_params(axis="x", labelrotation=rotation)
        for label in ax.get_xticklabels():
            label.set_horizontalalignment("right")

    plt.tight_layout()

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo"],
    percent_top=None,
    log1p=False,
    inplace=True
)

adata

In [ ]:
sc.pl.violin(
    adata,
    keys=["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="sample_id",
    jitter=0.2,
    multi_panel=True,
    show=False
)

rotate_x_labels(rotation=45)

plt.show()

In [ ]:
qc_summary = (
    adata.obs
    .groupby("sample_id")[["n_genes_by_counts", "total_counts", "pct_counts_mt"]]
    .quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
)

qc_summary

In [ ]:
sns.lmplot(
    data=adata.obs,
    x="total_counts",
    y="pct_counts_mt",
    col="sample_id",
    hue="sample_id",
    fit_reg=False,
    scatter_kws={"s": 3, "alpha": 0.4},
    col_wrap=2,
    height=4
)

plt.show()

In [ ]:
sc.pl.scatter(
    adata,
    x="total_counts",
    y="pct_counts_mt",
    color="sample_id"
)

In [ ]:
MAX_PCT_MT = 20
MIN_GENES = 500
MIN_COUNTS = 1000
MAX_GENES = 6000

In [ ]:
qc_filter = (
    (adata.obs["pct_counts_mt"] < MAX_PCT_MT) &
    (adata.obs["n_genes_by_counts"] >= MIN_GENES) &
    (adata.obs["total_counts"] >= MIN_COUNTS) &
    (adata.obs["n_genes_by_counts"] <= MAX_GENES)
)

adata_qc = adata[qc_filter].copy()

adata_qc

In [ ]:
n_cells_before = adata.n_obs
n_cells_after = adata_qc.n_obs
n_cells_removed = n_cells_before - n_cells_after
pct_removed = n_cells_removed / n_cells_before * 100

print(f"Cells before QC: {n_cells_before}")
print(f"Cells after QC:  {n_cells_after}")
print(f"Cells removed:   {n_cells_removed} ({pct_removed:.2f}%)")

In [ ]:
qc_cell_counts = pd.DataFrame({
    "before_qc": adata.obs["sample_id"].value_counts(),
    "after_qc": adata_qc.obs["sample_id"].value_counts()
})

qc_cell_counts["removed"] = qc_cell_counts["before_qc"] - qc_cell_counts["after_qc"]
qc_cell_counts["removed_pct"] = (
    qc_cell_counts["removed"] / qc_cell_counts["before_qc"] * 100
)

qc_cell_counts

In [ ]:
sc.pl.violin(
    adata_qc,
    keys=["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="sample_id",
    jitter=0.4,
    multi_panel=True,
    show=False
)

rotate_x_labels(rotation=45)

plt.show()

In [ ]:
qc_summary.to_csv(TABLES_DIR / "qc_summary_before_filtering.csv")
qc_cell_counts.to_csv(TABLES_DIR / "qc_cell_counts_before_after_filtering.csv")

In [ ]:
adata_qc.write_h5ad(
    ANNDATA_DIR / "galietta_airway_qc_filtered.h5ad"
)

In [ ]:
# Store raw counts before normalization
adata_qc.layers["counts"] = adata_qc.X.copy()

# Normalize total counts per cell
sc.pp.normalize_total(
    adata_qc,
    target_sum=1e4
)

sc.pp.log1p(adata_qc)

adata_qc.raw = adata_qc.copy()

adata_qc

In [ ]:
import numpy as np


def check_expression_layers(adata, counts_layer="counts", n_values=10):
    """
    Check raw counts are stored in layers[counts_layer]
    and if X contains normalized/log-transformed values.
    """
    if counts_layer not in adata.layers:
        raise ValueError(f"Layer '{counts_layer}' not found in adata.layers")

    X = adata.X
    counts = adata.layers[counts_layer]

    print("Object shape:")
    print(adata.shape)

    print("\nX matrix:")
    print("min:", X.min())
    print("max:", X.max())
    print("mean:", X.mean())

    print(f"\nLayer '{counts_layer}':")
    print("min:", counts.min())
    print("max:", counts.max())
    print("mean:", counts.mean())

    print("\nFirst values from X:")
    print(X[:1, :n_values].toarray() if hasattr(X, "toarray") else X[:1, :n_values])

    print(f"\nFirst values from layer '{counts_layer}':")
    print(
        counts[:1, :n_values].toarray()
        if hasattr(counts, "toarray")
        else counts[:1, :n_values]
    )

    # Check if counts look integer-like
    counts_values = counts.data if hasattr(counts, "data") else np.asarray(counts).ravel()
    is_integer_like = np.allclose(counts_values[:1000], np.round(counts_values[:1000]))

    print(f"\nCounts layer integer-like: {is_integer_like}")

check_expression_layers(adata_qc, counts_layer="counts")

In [ ]:
sc.pp.highly_variable_genes(
    adata_qc,
    flavor="seurat_v3",
    n_top_genes=3000,
    layer="counts",
    batch_key="sample_id",
    subset=False
)

In [ ]:
adata_qc.var["highly_variable"].value_counts()


adata_hvg = adata_qc[:, adata_qc.var["highly_variable"]].copy()

adata_hvg


In [ ]:
sc.pp.scale(
    adata_hvg,
    max_value=10

)
#Check scaling data 
N_PCS = 20
RANDOM_STATE = 0

sc.tl.pca(
    adata_hvg,
    svd_solver="arpack",
    random_state=RANDOM_STATE
)


sc.pl.pca_variance_ratio(
    adata_hvg,
    n_pcs=50,
    log=True
)



In [ ]:
N_NEIGHBORS = 30
N_PCS = 20
LEIDEN_RESOLUTION = 0.5
LEIDEN_KEY = "leiden_nn30_res_0_5"


In [ ]:
sc.pp.neighbors(
    adata_hvg,
    n_neighbors=N_NEIGHBORS,
    n_pcs=N_PCS
)

In [ ]:
sc.tl.umap(
    adata_hvg,
    random_state=RANDOM_STATE
)

sc.tl.leiden(
    adata_hvg,
    resolution=LEIDEN_RESOLUTION,
    key_added=LEIDEN_KEY,
    random_state=RANDOM_STATE
)

In [ ]:
sc.pl.umap(
    adata_hvg,
    color=[
        "sample_id",
        "condition",
        "cf_status",
        "treatment",
        LEIDEN_KEY,
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt",
    ],
    wspace=0.4,
    ncols=3
)


In [ ]:
def transfer_hvg_results_to_full_object(adata_full, adata_hvg, leiden_key):
    """
    Transfer PCA, UMAP, neighbors graph and Leiden clusters
    from the HVG object to the full QC-filtered object.
    """
    adata_full.obsm["X_umap"] = adata_hvg.obsm["X_umap"].copy()
    adata_full.obsm["X_pca"] = adata_hvg.obsm["X_pca"].copy()

    adata_full.obsp["distances"] = adata_hvg.obsp["distances"].copy()
    adata_full.obsp["connectivities"] = adata_hvg.obsp["connectivities"].copy()

    adata_full.uns["neighbors"] = adata_hvg.uns["neighbors"].copy()
    adata_full.uns["umap"] = adata_hvg.uns["umap"].copy()
    adata_full.uns["pca"] = adata_hvg.uns["pca"].copy()

    adata_full.obs[leiden_key] = adata_hvg.obs[leiden_key].copy()

    return adata_full


adata_qc = transfer_hvg_results_to_full_object(
    adata_full=adata_qc,
    adata_hvg=adata_hvg,
    leiden_key=LEIDEN_KEY
)

adata_qc

In [ ]:
sc.pl.umap(
    adata_qc,
    color=[
        "sample_id",
        "condition",
        "cf_status",
        "treatment",
        LEIDEN_KEY,
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt",
    ],
    wspace=0.4,
    ncols=3
)


In [ ]:
### Starting annotation. Reporting the final choices
marker_panel = {
    "Club Cells": [
        "SCGB1A1", "SCGB3A1", "WFDC2"
    ],

    "Goblet-like Cells": [
        "AGR2", "MUC5B", "MUC5AC", "WFDC2", "FCGBP", "PIGR"
    ],

    "Multiciliated Epithelial Cells": [
        "DNAH5", "DNAAF1", "RSPH1", "RSPH4A", "CDC20B", "CCNO", "DEUP1"
    ],

    "Interferon-Stimulated Epithelial Cells": [
        "IFI6", "ISG15", "IFI44L", "MX1", "MX2"
    ],

    "Basal Epithelial Cells": [
        "KRT5", "KRT15", "KRT17", "MKI67", "TOP2A", "NUSAP1"
    ],

    "Inflammatory-Stimulated Epithelial Cells": [
        "CXCL1", "CXCL6", "CCL20", "S100A8", "S100A9", "CXCL8", "KRT14", "KRT6A"
    ],

    "Pulmonary Ionocytes": [
        "FOXI1", "ASCL3", "KIT", "CFTR"
    ],
}


In [ ]:
def filter_marker_panel(adata, marker_panel, use_raw=True):
    """
    Keep only marker genes present in the AnnData object.
    If use_raw=True, markers are checked against adata.raw.var_names.
    """
    if use_raw:
        if adata.raw is None:
            raise ValueError("adata.raw is None, but use_raw=True")
        var_names = adata.raw.var_names
    else:
        var_names = adata.var_names

    marker_panel_present = {}
    missing_markers = {}

    for cell_type, genes in marker_panel.items():
        present = [gene for gene in genes if gene in var_names]
        missing = [gene for gene in genes if gene not in var_names]

        if len(present) > 0:
            marker_panel_present[cell_type] = present

        if len(missing) > 0:
            missing_markers[cell_type] = missing

    print("Marker present:")
    for cell_type, genes in marker_panel_present.items():
        print(f"{cell_type}: {genes}")

    print("\nMissing markers:")
    for cell_type, genes in missing_markers.items():
        print(f"{cell_type}: {genes}")

    return marker_panel_present, missing_markers


marker_panel_present, missing_markers = filter_marker_panel(
    adata_qc,
    marker_panel,
    use_raw=True
)


sc.pl.dotplot(
    adata_qc,
    marker_panel_present,
    groupby=LEIDEN_KEY,
    use_raw=True,
    standard_scale="var",
    dendrogram=False,
    figsize=(16, 9),
    swap_axes=True,
    cmap="viridis",
    dot_max=0.9,
    dot_min=0.0,
    show=True
)




In [ ]:
curated_coarse_cell_type_map = {
    "0": "Club Cells",
    "1": "Multiciliated Epithelial Cells",
    "2": "IFN-Stimulated Club Cells",
    "3": "Basal Epithelial Cells",
    "4": "Goblet-like Cells",
    "5": "Basal Epithelial Cells",
    "6": "IFN-Stimulated Club/Ciliated Transitional Cells",
    "7": "Multiciliated Epithelial Cells",
    "8": "IFN-Stimulated Basal Epithelial Cells",
    "9": "Basal Epithelial Cells",
    "10": "Basal Epithelial Cells",
    "11": "Pulmonary Ionocytes",
    "12": "Deuterosomal/Ciliating Cells",
}

curated_cell_type_map = {
    "0": "Club cell",
    "1": "Mature multiciliated epithelial cells",
    "2": "IFN-stimulated club cells",
    "3": "Quiescent basal epithelial progenitors",
    "4": "Inflammatory goblet-like epithelial cells",
    "5": "Cycling basal epithelial cells",
    "6": "IFN-stimulated ciliary-primed epithelial cells",
    "7": "Multiciliated epithelial cells (TNFα/IL-17 stimulated)",
    "8": "IFN-stimulated basal epithelial cells",
    "9": "Inflammatory basal epithelial cells",
    "10": "Squamous metaplasia basal cells",
    "11": "Mature pulmonary ionocyte",
    "12": "Deuterosomal multiciliated epithelial cells",
}

adata_qc.obs["curated_coarse_cell_type"] = (
    adata_qc.obs[LEIDEN_KEY]
    .astype(str)
    .map(curated_coarse_cell_type_map)
    .astype("category")
)

adata_qc.obs["curated_cell_type"] = (
    adata_qc.obs[LEIDEN_KEY]
    .astype(str)
    .map(curated_cell_type_map)
    .astype("category")
)

print("curated_coarse_cell_type")
display(pd.crosstab(
    adata_qc.obs[LEIDEN_KEY],
    adata_qc.obs["curated_coarse_cell_type"]
))

print("curated_cell_type")
display(pd.crosstab(
    adata_qc.obs[LEIDEN_KEY],
    adata_qc.obs["curated_cell_type"]
))

In [ ]:
sc.pl.dotplot(
    adata_qc,
    marker_panel_present,
    groupby="curated_coarse_cell_type",
    use_raw=True,
    standard_scale="var",
    dendrogram=False,
    figsize=(16, 9),
    swap_axes=True,
    cmap="viridis",
    dot_max=0.9,
    dot_min=0.0,
    show=True
)

In [ ]:
curated_palette = {
    "Club Cells": "#1f77b4",                                      # blue
    "Multiciliated Epithelial Cells": "#ffb000",                  # amber/yellow
    "Deuterosomal/Ciliating Cells": "#e6ab02",                    # darker yellow/ochre
    "Basal Epithelial Cells": "#cc79a7",                          # purple/pink
    "Goblet-like Cells": "#2ca02c",                               # green
    "IFN-Stimulated Basal Epithelial Cells": "#17becf",           # cyan
    "IFN-Stimulated Club Cells": "#6baed6",                       # light blue
    "IFN-Stimulated Club/Ciliated Transitional Cells": "#00a087", # teal
    "Pulmonary Ionocytes": "#d55e00",                             # vermillion/orange-red
}

coarse_key = "curated_coarse_cell_type"

adata_qc.obs[coarse_key] = adata_qc.obs[coarse_key].astype("category")

missing_colors = set(adata_qc.obs[coarse_key].cat.categories) - set(curated_palette)
if missing_colors:
    raise ValueError(f"Missing colors for categories: {missing_colors}")

adata_qc.uns[f"{coarse_key}_colors"] = [
    curated_palette[cat]
    for cat in adata_qc.obs[coarse_key].cat.categories
]

In [ ]:
sc.pl.umap(
    adata_qc,
    color=coarse_key,
    legend_loc="right margin",
    frameon=False,
    title="Curated coarse cell types"
)

In [ ]:
##No treated
genes_to_plot = [
    "TRPV4",
    "CFTR",
    "PANX1",
    "ANO1"
]


In [ ]:
sample_key = "sample_id"
sample_of_interest = "BE63_NON_CF_NT"

adata_nt = adata_qc[
    adata_qc.obs[sample_key] == sample_of_interest
].copy()

adata_nt


def transfer_category_colors(adata_source, adata_target, category_key):
    
    #  Transfer category colors from a source AnnData object to a subset AnnData object
    color_key = f"{category_key}_colors"

    if category_key not in adata_source.obs:
        raise KeyError(f"'{category_key}' not found in source adata.obs")

    if category_key not in adata_target.obs:
        raise KeyError(f"'{category_key}' not found in target adata.obs")

    if color_key not in adata_source.uns:
        raise KeyError(f"'{color_key}' not found in source adata.uns")

    adata_target.obs[category_key] = (
        adata_target.obs[category_key]
        .astype("category")
        .cat.remove_unused_categories()
    )

    source_categories = adata_source.obs[category_key].cat.categories
    source_colors = adata_source.uns[color_key]

    color_map = dict(zip(source_categories, source_colors))

    adata_target.uns[color_key] = [
        color_map[cat]
        for cat in adata_target.obs[category_key].cat.categories
    ]

    return adata_target

coarse_key = "curated_coarse_cell_type"

adata_nt = transfer_category_colors(
    adata_source=adata_qc,
    adata_target=adata_nt,
    category_key=coarse_key
)

def compare_category_colors(adata_a, adata_b, category_key, label_a="adata_a", label_b="adata_b"):
    
    #Compare category colors between two AnnData objects.
    
    color_key = f"{category_key}_colors"

    colors_a = dict(zip(
        adata_a.obs[category_key].cat.categories,
        adata_a.uns[color_key]
    ))

    colors_b = dict(zip(
        adata_b.obs[category_key].cat.categories,
        adata_b.uns[color_key]
    ))

    common_categories = set(colors_a).intersection(colors_b)

    for cat in sorted(common_categories):
        color_a = colors_a[cat]
        color_b = colors_b[cat]

        if color_a != color_b:
            print(f"{cat}: different")
            print(f"  {label_a}: {color_a}")
            print(f"  {label_b}: {color_b}")
        else:
            print(f"{cat}: same ({color_a})")

compare_category_colors(
    adata_a=adata_qc,
    adata_b=adata_nt,
    category_key=coarse_key,
    label_a="adata_qc",
    label_b="adata_nt"
)




In [ ]:
sc.pl.umap(
    adata_nt,
    color=coarse_key,
    legend_loc="right margin",
    frameon=False,
    title=f"Curated coarse cell types - {sample_of_interest}",
    show=True
)

In [ ]:
genes_present = [
    gene for gene in genes_to_plot
    if gene in adata_nt.var_names
]

genes_missing = [
    gene for gene in genes_to_plot
    if gene not in adata_nt.var_names
]

print("Genes present:", genes_present)
print("Genes missing:", genes_missing)

In [ ]:
sc.pl.dotplot(
    adata_nt,
    var_names=genes_present,
    groupby=coarse_key,
    use_raw=True,
    standard_scale="var",
    dendrogram=False,
    figsize=(8, 4),
    title=f"Expression in {sample_of_interest}",
    show=True
)